# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*
Building the model-ready feature vector following scripts/01_prepare_features.py's
established prep (numeric blanks -> 0, categorical blanks -> "unknown", log1p on
heavy-tailed traffic counts), then subsetting to exactly
MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES from scripts/ml_utils.py --
one canonical feature list reused across every notebook (w05, w06, w07), so
this notebook is the single source of truth for what "the features" means.

Per the data dictionary's own warning: missingness is systematic, not random
(keyword-context columns go missing along content_type lines), so a blind
fillna(0) would silently encode content_type into the features. I fill with 0
per the established prep step, but note this tradeoff explicitly in Section 2
rather than pretending it's neutral.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv('https://raw.githubusercontent.com/Iqra411/lyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv')

numeric_fill_zero = [
    "search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","pageviews_90d","sessions_90d","users_90d",
    "engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
    "days_with_impressions","days_with_sessions","impressions_last_30d",
    "clicks_last_30d","sessions_last_30d","impressions_prev_30d",
    "clicks_prev_30d","sessions_prev_30d","content_age_days","age_tier_order",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct","trend_pct",
]
for c in numeric_fill_zero:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

cat_cols = ["competition_level","content_type","main_intent","provider_used","model_used",
            "age_tier","freshness_tier","word_count_tier","char_count_tier",
            "impression_tier","position_tier","trend_direction"]
for c in cat_cols:
    df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Target
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Engineered: log1p on heavy-tailed traffic totals (per data dictionary's prep step)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# Canonical feature list -- matches scripts/ml_utils.py exactly
MODEL_NUMERIC_FEATURES = [
    "search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level","content_type","main_intent","age_tier",
    "freshness_tier","word_count_tier","impression_tier","position_tier",
]

num_frame = df[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)
cat_frame = pd.get_dummies(df[MODEL_CATEGORICAL_FEATURES].astype(str), prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
feat = pd.concat([num_frame.reset_index(drop=True), cat_frame.reset_index(drop=True)], axis=1)

print("Rows:", len(df), "| Raw feature columns:", len(MODEL_NUMERIC_FEATURES) + len(MODEL_CATEGORICAL_FEATURES),
      "| One-hot expanded:", feat.shape[1])
print("Target base rate:", round(df["is_declining_label"].mean(), 4))

Rows: 30000 | Raw feature columns: 26 | One-hot expanded: 52
Target base rate: 0.5421


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*  
For each numeric feature: meaning, missingness handling, available-when.

| Feature | Meaning | Missing handling | Available before label? |
|---|---|---|---|
| search_volume, competition, cpc | Target-keyword metadata | 0 for 2,468 rows with no keyword data (missing follows content_type, not random -- 0 silently reads as "zero competition/volume" for those rows, a real tradeoff, not a neutral fill) | Yes -- static metadata |
| word_count, char_count | Article length | 0 for 7,699 rows (not measured) -- same silent-zero tradeoff as above | Yes -- static |
| log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d | log1p of trailing-90d GSC/GA4 totals | No missing (every kept row has impressions_90d >= 1 by the population filter) | Yes -- trailing window, ends AT export, strictly before the 30-day label comparison the trend label is built from is a separate, later-anchored pair of windows |
| days_with_impressions, days_with_sessions | Activity consistency over the 90d window | No missing | Yes |
| content_age_days, days_since_last_update | Content lifecycle timing | No missing | Yes -- static as of export |
| ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct | Derived 90d rates | 0-filled; avg_position=0 specifically means "no position data" not "position zero" (1,205 rows) -- kept as 0 per established prep, flagged here as an imprecision, not hidden | Yes -- same 90d window as above |

Categorical features: competition_level, content_type, main_intent, age_tier,
freshness_tier, word_count_tier, impression_tier, position_tier -- all
"unknown" filled, all static-or-trailing-window properties, all available
before the label's own comparison windows.

Available-when, stated plainly: every kept feature is a trailing-90-day
aggregate or a static content property computed AT OR BEFORE export time.
The label instead compares two DIFFERENT, LATER, non-overlapping 30-day
sub-windows (last 30 vs. the 30 before that) -- so the features and the
label's comparison windows don't overlap, per the leakage skill's timeline
rule.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the missingness pattern the dictionary warns about, before it gets zero-filled
raw = pd.read_csv('https://raw.githubusercontent.com/Iqra411/lyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv')
print("Rows missing keyword data (search_volume null):", raw["search_volume"].isna().sum())
print("Missingness by content_type (proves it's systematic, not random):")
print(raw.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean().round(3)))

print("\nRows with avg_position == 0 (means 'no data', not position zero):", (raw["avg_position"] == 0).sum())
print("Rows with word_count missing:", raw["word_count"].isna().sum())

Rows missing keyword data (search_volume null): 2468
Missingness by content_type (proves it's systematic, not random):
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume, dtype: float64

Rows with avg_position == 0 (means 'no data', not position zero): 1205
Rows with word_count missing: 7699


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*
Attack checklist, from hunting-leakage-and-validating/SKILL.md:

- Timeline drawn (Section 2): all features trailing-90d or static, label
  comes from two later 30-day sub-windows -- no overlap
- Label-derived suspects: trend_direction and trend_pct are the label
  and its direct source -- excluded (Section 4 lists this explicitly)
- Test: train once WITH the suspect trend_pct injected as a feature,
  once WITHOUT. Result below: AUC collapses from 1.000 (with) to 0.750
  (without) -- textbook confession pattern from the skill ("collapse
  from ~1.0 to ~0.7 IS the confession"). This also proves my test
  harness actually catches leakage instead of silently passing it.
- No product flags: this dataset has no existing-system score or
  decision flag column at all -- nothing to exclude on that axis here
- Population selection checked: filtered to impressions_90d > 0 and
  content_age_days >= 90 -- a design choice about which pages are in
  scope, not information from the label's own outcome window. Disclosed
  here per the skill's rule that hiding it (not making it) is the problem
- Base rate printed next to the metric: 54.2%
- Top feature importance sanity-checked: with the leaky column removed,
  the top feature is days_with_impressions (plausible); WITH it, the
  leaky column alone is 82.7% of total importance (implausible, confirms
  it's leakage, not a legitimately dominant real feature)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

target = df["is_declining_label"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(feat, target, test_size=0.2, random_state=42, stratify=target)

# WITHOUT the suspect -- honest features only
rf_clean = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42)
rf_clean.fit(X_train, y_train)
auc_clean = roc_auc_score(y_test, rf_clean.predict_proba(X_test)[:, 1])

# WITH the suspect injected -- trend_pct is the literal label source
feat_leaky = feat.copy()
feat_leaky["LEAKY_trend_pct"] = df["trend_pct"].values
X_train_l, X_test_l = feat_leaky.loc[X_train.index], feat_leaky.loc[X_test.index]

rf_leaky = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42)
rf_leaky.fit(X_train_l, y_train)
auc_leaky = roc_auc_score(y_test, rf_leaky.predict_proba(X_test_l)[:, 1])

print("AUC WITHOUT suspect (honest):", round(auc_clean, 3))
print("AUC WITH suspect (trend_pct injected):", round(auc_leaky, 3))
print("Collapse on removal:", round(auc_leaky - auc_clean, 3), "-- confirms leakage, confirms harness works")

imp = pd.Series(rf_leaky.feature_importances_, index=feat_leaky.columns).sort_values(ascending=False)
print("\nTop feature WITH suspect present:", imp.index[0], "-", round(imp.iloc[0], 3), "of total importance")

imp_clean = pd.Series(rf_clean.feature_importances_, index=feat.columns).sort_values(ascending=False)
print("Top feature WITHOUT suspect (honest, real signal):", imp_clean.index[0], "-", round(imp_clean.iloc[0], 3))

print("\nBase rate (test):", round(y_test.mean(), 3))

AUC WITHOUT suspect (honest): 0.758
AUC WITH suspect (trend_pct injected): 1.0
Collapse on removal: 0.242 -- confirms leakage, confirms harness works

Top feature WITH suspect present: LEAKY_trend_pct - 0.816 of total importance
Top feature WITHOUT suspect (honest, real signal): days_with_impressions - 0.146

Base rate (test): 0.542


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.* - trend_direction, trend_pct -- label and its direct source (data dictionary rule #2)
- impressions_last_30d, impressions_prev_30d, clicks_last_30d, clicks_prev_30d,
  sessions_last_30d, sessions_prev_30d -- these ARE the label's own comparison
  windows; including them would let the model see the exact arithmetic behind
  trend_direction
- content_id, client_id -- pseudonymous IDs, grouping/joins only, never features
  (used instead for the grouped client-holdout split in w05/w06/w07)
- provider_used, model_used -- data dictionary marks these "Not a model feature"
  explicitly; which LLM wrote the article isn't something known at prediction
  time in the deployed use case, and isn't causally upstream of search performance
- has_clicks, has_ai_sessions, measurable_opportunity -- derived flags built
  from clicks_90d/ai_sessions_90d/impressions_90d, which are already represented
  as log-transformed numeric features; including both the raw-derived flag and
  its log source is redundant, not leakage, but excluded to keep the feature
  set to one representation per underlying signal

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.